In [0]:
%pip install newsapi-python==0.2.7

# Import necessary libraries
from datetime import date, timedelta
from newsapi.newsapi_client import NewsApiClient
import pandas as pd
import logging

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Reinitialize the API client
NEWS_API_KEY = ''
newsapi = NewsApiClient(api_key=NEWS_API_KEY)

def extract_news():
    top_headlines = newsapi.get_top_headlines(
        category='entertainment',
        language='en',
        page_size=90,
        page=1
    )

    articles = top_headlines.get('articles', [])
    if not articles:
        logger.warning("No articles retrieved.")
        return pd.DataFrame()

    df = pd.DataFrame(articles, columns=['source', 'title', 'publishedAt', 'author', 'url'])
    df['source'] = df['source'].apply(lambda x: x['name'] if pd.notna(x) and 'name' in x else None)
    df['publishedAt'] = pd.to_datetime(df['publishedAt'])
    df.rename(columns={'publishedAt': 'date_posted'}, inplace=True)
    
    return df

dataframe = extract_news()

# Save extracted data to a CSV file for the next notebook using dbutils.fs.put
#csv_content = dataframe.to_csv(index=False)
#dbutils.fs.put("/dbfs/tmp/extracted_news.csv", csv_content)
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("NewsProcessing").getOrCreate()
spark_df = spark.createDataFrame(dataframe)
spark_df.write.format("delta").mode('overwrite').save("/mnt/data/extracted_news")

Python interpreter will be restarted.
Python interpreter will be restarted.
